# Task 1: Teaching NanoGPT to Do Math

Team contributions: Task 1 - fill in your name; Tasks 2 and 3 - fill in teammates' names. AI assistance: OpenAI Codex assisted with dataset design, code, explanations and validation. We reviewed the implementation and take responsibility for the submitted work. Our professor explicitly permits AI-generated code, superseding the PDF restriction.

This section runs sequentially from the project root or `dpo/`. It requires only Python's standard library for synthetic generation. Optional model generation requires PyTorch. All implementation code is included here. The supplied Steps 1-8 follow below, guarded by `RUN_TASKS_2_3=False`. Steps 5-7 (Task 2, DPO training) are now complete; Step 8 (Task 3, testing) remains for a teammate. No model or configurator source was changed.

## Design and distribution

Default: 100,000 pairs, 20,000 each for addition, subtraction, multiplication, exact division and one-variable algebra. Addition/subtraction operands are 0-199, multiplication operands 0-99, division is constructed from a denominator 1-99 and quotient 0-99. Algebra uses nonzero coefficients/constants 1-99 and integer variables/quotients -99 to 99; forms are `x+a=b`, `x-a=b`, `a-x=b`, `a*x=b`, and `x/a=b`. The largest division dividend and algebra product can reach 9,801. These are modest integer problems, with some harder four-digit results for generalization; adjust the ranges in `pool` if desired. The supplied examples use direct arithmetic and inverse operations. We retain that style.

Three compact spacing variants add modest surface variation. Distinct questions can share the same mathematical calculation; they are not identical pairs. We sample without replacement, reserve development calculation groups before sampling, and prohibit overlap even after whitespace normalization. This reserves 500 development pairs separately. Hold out additional unseen calculations for final Task 3 evaluation; development results are not an unbiased final test if used to tune choices.

Every positive gives an integer and one correct arithmetic equality. For algebra the equality performs the inverse operation. An independent Fraction-based parser verifies both the equation's solution and the explanation. Every full response plus four newline terminators fits the supplied `max_length=64`, with no truncation. The checkpoint header is inspected directly to obtain its actual context limit without requiring PyTorch. Padding feeds 63 input positions to the model; this must fit the checkpoint context.

## Negatives and tradeoffs

The default dataset is entirely synthetic: approximately 90% explicit wrong integer answers (operation confusion, sign errors and small offsets), and 10% varied unhelpful responses. Candidates equal to the correct answer are discarded. This is cheap, reproducible and auditable, but does not reproduce the pretrained model's error distribution. Origins and categories are recorded in a separate metadata file to preserve exactly the sample's two-key schema.

Set `MODEL_FRACTION=0.1` to attempt pretrained negatives for 10% of training questions. The optional path uses the supplied `GPT`, `GPTConfig`, `model_args`, `model` state dictionary, prefix cleanup and `generate` interface. It batches equal-length prompts, uses CUDA/MPS/CPU, retries three times and atomically checkpoints each batch. Rejected attempts fall back to labeled synthetic responses. Thus the requested fraction is an attempt fraction, not a guaranteed accepted model fraction. A fingerprint covers questions, settings and checkpoint contents; resume only with identical settings/backend/library versions. GPU sampling reproducibility across hardware is not guaranteed.

Automatic free-text checking is unreliable: a response can mention the correct number incidentally, contradict itself, or explain a solution without a standard answer phrase. We deliberately accept only a complete newline-delimited single wrong integer answer or one of three exact unhelpful phrases. All other model text, including correct answers and partial continuations, is rejected. This trades low acceptance for reliable labels. Manual audit is still valuable for assessing instructional usefulness. No synthetic response is attributed to NanoGPT.

## Configuration, generation and independent validation

Change `DATASET_SIZE` (at least 10,000, divisible by five), `SEED`, ranges, or `MODEL_FRACTION` below. For model mode install PyTorch for your hardware first. The supplied pickle/checkpoint files are treated as trusted project inputs. Run all Task 1 cells, inspect the smoke outputs, then review the persisted validation report. Generation overwrites the named dataset; save a copy before changing settings. A conflicting model cache must be renamed before a different model run.

In [1]:
from pathlib import Path
from collections import Counter, OrderedDict, defaultdict
from fractions import Fraction
import hashlib, io, json, pickle, random, re, sys, zipfile

# Run from either the project root or its dpo folder.
ROOT = Path.cwd().resolve()
if not (ROOT / 'sft/meta.pkl').exists():
    ROOT = ROOT.parent
assert (ROOT / 'sft/meta.pkl').exists(), 'Run from project root or dpo/'
OUT = ROOT / 'dpo'
SEED = 3000
DATASET_SIZE = 100_000
DEV_PER_CATEGORY = 100
MODEL_FRACTION = 0.0  # Optional: 0.1 requests 10% model attempts; fallback is explicit.
MODEL_BATCH_SIZE = 16
MODEL_RETRIES = 3
MAX_LENGTH = 64  # Supplied get_batches appends four newlines before padding.
TERMINATOR = '\n\n\n\n'
CATEGORIES = ['addition', 'subtraction', 'multiplication', 'division', 'algebra']
assert DATASET_SIZE >= 10_000 and DATASET_SIZE % 5 == 0
assert 0 <= MODEL_FRACTION <= 1
with (ROOT / 'sft/meta.pkl').open('rb') as f:
    meta = pickle.load(f)
stoi, itos = meta['stoi'], meta['itos']

def checkpoint_args(path):
    # Read only model_args without importing torch or materializing tensors.
    # Reject unexpected globals; tensor rebuild functions return placeholders.
    class HeaderReader(pickle.Unpickler):
        def find_class(self, module, name):
            if (module, name) == ('collections', 'OrderedDict'):
                return OrderedDict
            if module == 'torch' and name.endswith('Storage'):
                return name
            if module == 'torch._utils' and name.startswith('_rebuild'):
                return lambda *args: None
            raise ValueError(f'Unexpected checkpoint global: {module}.{name}')
        def persistent_load(self, value):
            return None
    with zipfile.ZipFile(path) as z:
        names = [n for n in z.namelist() if n.endswith('/data.pkl')]
        assert len(names) == 1
        header = HeaderReader(io.BytesIO(z.read(names[0]))).load()
    return header['model_args']

MODEL_ARGS = checkpoint_args(ROOT / 'sft/gpt.pt')
assert MODEL_ARGS['vocab_size'] == meta['vocab_size'] == len(stoi)
# Training forwards length 63 after the notebook's 64-character padding.
assert MAX_LENGTH - 1 <= MODEL_ARGS['block_size']
print('Checkpoint configuration:', MODEL_ARGS)



Checkpoint configuration: {'n_layer': 6, 'n_head': 6, 'n_embd': 348, 'block_size': 256, 'bias': False, 'vocab_size': 74, 'dropout': 0.2}


### Question pools, positives and synthetic negatives

In [2]:
# Build finite pools, then sample without replacement. Formatting is limited
# to three familiar variants; we never repeat an identical question.
def formats(expression, algebra=False):
    suffix = ',x=?' if algebra else '=?'
    core = expression + suffix
    return [core, core.replace('=', ' = '), core.replace('=', ' =')]

def make_spec(category, q, answer, expression):
    return dict(category=category, question=q, answer=answer, expression=expression)

def pool(category):
    result = []
    if category in ('addition', 'subtraction', 'multiplication', 'division'):
        upper = 200 if category in ('addition', 'subtraction') else 100
        for a in range(upper):
            for b in range(upper):
                if category == 'addition': expr, ans = f'{a}+{b}', a+b
                elif category == 'subtraction': expr, ans = f'{a}-{b}', a-b
                elif category == 'multiplication': expr, ans = f'{a}*{b}', a*b
                else:
                    if a == 0: continue
                    expr, ans = f'{a*b}/{a}', b
                for q in formats(expr): result.append(make_spec(category, q, ans, expr))
    else:
        for x in range(-99, 100):
            for a in range(1, 100):
                # Unique solutions; inverse operation is the worked explanation.
                cases = [(f'x+{a}={x+a}', f'{x+a}-{a}'),
                         (f'x-{a}={x-a}', f'{x-a}+{a}'),
                         (f'{a}-x={a-x}', f'{a}-{a-x}'),
                         (f'{a}*x={a*x}', f'{a*x}/{a}'),
                         (f'x/{a}={x}', f'{x}*{a}')]
                for eq, inverse in cases:
                    answer = a*x if eq.startswith('x/') else x
                    for q in formats(eq, True): result.append(make_spec(category, q, answer, inverse))
    return result

def positive(spec):
    a, e = spec['answer'], spec['expression']
    return spec['question'] + f' The answer is {a} because {e}={a}.'

def compatible(text):
    return set(text) <= set(stoi) and len(text + TERMINATOR) <= MAX_LENGTH

def split_specs(n, dev_n):
    train, dev = [], []
    rng = random.Random(SEED)
    for cat in CATEGORIES:
        candidates = [s for s in pool(cat) if compatible(positive(s))]
        # Keep all spacing variants of a mathematical question in one split.
        groups = defaultdict(list)
        for s in candidates: groups[re.sub(r'\s+', '', s['question'])].append(s)
        keys = sorted(groups)
        rng.shuffle(keys)
        reserved = set(keys[:dev_n])
        dev.extend(groups[k][0] for k in keys[:dev_n])
        available = [s for k in keys if k not in reserved for s in groups[k]]
        assert len(available) >= n//5, (cat, len(available))
        train.extend(rng.sample(available, n//5))
    rng.shuffle(train)
    return train, dev

UNHELPFUL = ['Sorry, I do not know.', 'I cannot solve this.', 'Please ask someone else.']

def synthetic(spec, index):
    rng = random.Random(SEED + index)
    answer = spec['answer']
    if rng.random() < .1:
        return rng.choice(UNHELPFUL), 'unhelpful'
    # Prefer operation-confusion when it is wrong, else use a sign/offset error.
    expr = spec['expression']
    match = re.fullmatch(r'(-?\d+)([+*/-])(-?\d+)', expr)
    a, op, b = int(match[1]), match[2], int(match[3])
    wrong_ops = {'+': a-b, '-': a+b, '*': a+b, '/': a*b}
    choices = [(wrong_ops[op], 'wrong_operation'), (-answer, 'sign_error'),
               (answer + rng.choice([-2,-1,1,2]), 'offset_error')]
    choices = [(v,k) for v,k in choices if v != answer]
    wrong, kind = rng.choice(choices)
    return f'The answer is {wrong}.', kind

# Conservative acceptance: only a single explicit wrong integer answer or an
# exact unhelpful phrase is accepted. Arbitrary free text is rejected, even if
# it appears wrong: keyword/number matching cannot prove semantic incorrectness.
def accepted_negative(response, answer):
    if response in UNHELPFUL: return True
    match = re.fullmatch(r'(?:The answer is )?(-?\d+)\.?', response)
    return match is not None and int(match[1]) != answer



### Optional pretrained generation and atomic saving

In [3]:
MODEL_CACHE = OUT / 'task1_model_cache.json'

def model_responses(specs, indices):
    import torch
    sys.path.insert(0, str(ROOT))
    from model import GPT, GPTConfig
    device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    # Full state dict is loaded only here; the checkpoint must be trusted.
    ckpt = torch.load(ROOT / 'sft/gpt.pt', map_location=device, weights_only=False)
    model = GPT(GPTConfig(**ckpt['model_args']))
    state = {k.removeprefix('_orig_mod.'): v for k,v in ckpt['model'].items()}
    model.load_state_dict(state)
    model.to(device).eval()
    assert model.config.block_size == MODEL_ARGS['block_size']
    fingerprint = hashlib.sha256(json.dumps(dict(seed=SEED, specs=specs, indices=indices,
        args=MODEL_ARGS, retries=MODEL_RETRIES, batch=MODEL_BATCH_SIZE,
        checkpoint=hashlib.sha256((ROOT/'sft/gpt.pt').read_bytes()).hexdigest()), sort_keys=True).encode()).hexdigest()
    cache = {'fingerprint': fingerprint, 'results': {}}
    if MODEL_CACHE.exists():
        cache = json.loads(MODEL_CACHE.read_text())
        assert cache['fingerprint'] == fingerprint, 'Different run: rename old model cache first.'
    remaining = [i for i in indices if str(i) not in cache['results']]
    grouped = defaultdict(list)
    for i in remaining: grouped[len(specs[i]['question'])].append(i)
    done = len(indices) - len(remaining)
    for length, group in sorted(grouped.items()):
        for start in range(0, len(group), MODEL_BATCH_SIZE):
            batch = group[start:start+MODEL_BATCH_SIZE]
            # Seed per batch. Cache prevents regrouping completed work on resume.
            pending, found = batch[:], {}
            torch.manual_seed(SEED + batch[0])
            for attempt in range(MODEL_RETRIES):
                if not pending: break
                ids = torch.tensor([[stoi[c] for c in specs[i]['question']] for i in pending], device=device)
                with torch.no_grad():
                    generated = model.generate(ids, max_new_tokens=60-length,
                        temperature=.8, top_k=min(200, model.config.vocab_size))
                for i, row in zip(pending, generated.tolist()):
                    continuation = ''.join(itos[t] for t in row[length:])
                    # A newline delimits an actual QA response; never trim a
                    # length-exceeded candidate into an accepted response.
                    if '\n' not in continuation: continue  # Reject possibly cut-off text.
                    response = continuation.split('\n', 1)[0].strip()
                    if accepted_negative(response, specs[i]['answer']): found[i] = response
                pending = [i for i in pending if i not in found]
            for i in batch: cache['results'][str(i)] = found.get(i)
            atomic_json(MODEL_CACHE, cache)
            done += len(batch)
            print(f'Model attempts completed: {done}/{len(indices)} on {device}')
    return cache['results']

def atomic_json(path, value):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    tmp.replace(path)

def build(specs, use_model=False):
    indices = random.Random(SEED).sample(range(len(specs)), int(len(specs)*MODEL_FRACTION)) if use_model else []
    results = model_responses(specs, indices) if indices else {}
    pairs, provenance = [], []
    for i,s in enumerate(specs):
        response, kind = synthetic(s, i)
        origin = 'synthetic'
        if str(i) in results:
            if results[str(i)] is not None:
                response, kind, origin = results[str(i)], 'filtered_model', 'NanoGPT'
            else: origin = 'synthetic_after_model_rejection'
        neg = s['question'] + ' ' + response
        assert compatible(neg)
        pairs.append({'negative': neg, 'positive': positive(s)})
        provenance.append(dict(index=i, **s, origin=origin, negative_type=kind))
        if (i+1) % 20000 == 0: print(f'Constructed {i+1:,} pairs')
    return pairs, provenance



### Independent validators

In [4]:
# Independent arithmetic parser (Fractions, no eval) and question solver.
def evaluate(expr):
    m = re.fullmatch(r'(-?\d+)([+*/-])(-?\d+)', expr)
    assert m, expr
    a, op, b = Fraction(int(m[1])), m[2], Fraction(int(m[3]))
    if op == '+': return a+b
    if op == '-': return a-b
    if op == '*': return a*b
    assert b != 0
    return a/b

def solve_question(q):
    q = re.sub(r'\s+', '', q)
    if not q.endswith(',x=?'): return evaluate(q[:-2])
    left, right = q[:-4].split('=')
    rhs = Fraction(int(right))
    m = re.fullmatch(r'x([+/-])(\d+)', left)
    if m:
        b = int(m[2])
        return rhs-b if m[1]=='+' else rhs+b if m[1]=='-' else rhs*b
    m = re.fullmatch(r'(\d+)([-*])x', left)
    assert m, q
    a = int(m[1]); assert a != 0
    return a-rhs if m[2]=='-' else rhs/a

def validate(pairs, provenance, expected_counts, dev_questions=()):
    assert len(pairs) == len(provenance) == sum(expected_counts.values())
    questions, pair_keys, lengths = set(), set(), []
    dev_canonical = {re.sub(r'\s+', '', q) for q in dev_questions}
    for pair, info in zip(pairs, provenance):
        assert set(pair) == {'negative','positive'}
        assert all(isinstance(v,str) for v in pair.values())
        q = pair['positive'].split('?',1)[0]+'?'
        assert pair['negative'].split('?',1)[0]+'?' == q == info['question']
        assert q not in questions; questions.add(q)
        key = (pair['positive'],pair['negative']); assert key not in pair_keys; pair_keys.add(key)
        assert pair['positive'] != pair['negative']
        assert re.sub(r'\s+', '', q) not in dev_canonical
        answer = solve_question(q)
        assert answer.denominator == 1 and answer == info['answer']
        body = pair['positive'][len(q)+1:]
        m = re.fullmatch(r'The answer is (-?\d+) because (.+)=(-?\d+)\.', body)
        assert m and Fraction(int(m[1])) == answer == evaluate(m[2]) == Fraction(int(m[3]))
        assert accepted_negative(pair['negative'][len(q)+1:], int(answer))
        for text in pair.values():
            assert compatible(text), text
            lengths.append(len(text+TERMINATOR))
    counts = Counter(p['category'] for p in provenance)
    assert counts == Counter(expected_counts)
    return dict(pair_count=len(pairs), category_counts=dict(counts),
        origins=dict(Counter(p['origin'] for p in provenance)),
        negative_types=dict(Counter(p['negative_type'] for p in provenance)),
        max_length_including_four_newlines=max(lengths),
        duplicate_questions=0, duplicate_pairs=0, unsupported_characters=0,
        identical_responses=0, dev_overlap=0, model_context_limit=MODEL_ARGS['block_size'],
        context_validation='checkpoint header inspected; model forward not executed')


### Smoke test, full generation and reopening saved JSON

In [5]:
# Smoke test before full generation, and verify validators reject bad data.
smoke_specs, smoke_dev = split_specs(100, 5)
smoke_pairs, smoke_meta = build(smoke_specs)
print('SMOKE:', json.dumps(validate(smoke_pairs, smoke_meta, {c:20 for c in CATEGORIES}, [s['question'] for s in smoke_dev]), indent=2))
import copy
for failure in ['positive', 'correct_negative', 'question', 'duplicate', 'character', 'length']:
    broken = copy.deepcopy(smoke_pairs)
    if failure == 'positive': broken[0]['positive'] = broken[0]['positive'].replace('because', 'since')
    elif failure == 'correct_negative': broken[0]['negative'] = broken[0]['positive']
    elif failure == 'question': broken[0]['negative'] = 'Different=? Wrong.'
    elif failure == 'duplicate': broken[1] = broken[0]
    elif failure == 'character': broken[0]['negative'] += '🙂'
    else: broken[0]['negative'] += ' ' * 100
    try: validate(broken, smoke_meta, {c:20 for c in CATEGORIES})
    except AssertionError: print('Rejected smoke corruption:', failure)
    else: raise AssertionError('Validator failed to reject ' + failure)
print('Representative smoke pairs:', json.dumps(smoke_pairs[:3], indent=2))

train_specs, dev_specs = split_specs(DATASET_SIZE, DEV_PER_CATEGORY)
pairs, provenance = build(train_specs, use_model=True)
report = validate(pairs, provenance, {c:DATASET_SIZE//5 for c in CATEGORIES}, [s['question'] for s in dev_specs])
atomic_json(OUT / 'pos_neg_pairs.json', pairs)
atomic_json(OUT / 'task1_metadata.json', dict(seed=SEED, dataset_size=DATASET_SIZE, requested_model_fraction=MODEL_FRACTION,
    checkpoint_args=MODEL_ARGS, records=provenance))
dev_pairs, dev_meta = build(dev_specs)
validate(dev_pairs, dev_meta, {c:DEV_PER_CATEGORY for c in CATEGORIES})
atomic_json(OUT / 'task1_dev_pairs.json', dev_pairs)
# Reopen persisted deliverables rather than merely validate in-memory objects.
reopened = json.loads((OUT / 'pos_neg_pairs.json').read_text(encoding='utf-8'))
stored_meta = json.loads((OUT / 'task1_metadata.json').read_text(encoding='utf-8'))
report = validate(reopened, stored_meta['records'], {c:DATASET_SIZE//5 for c in CATEGORIES}, [s['question'] for s in dev_specs])
report['dataset_sha256'] = hashlib.sha256((OUT/'pos_neg_pairs.json').read_bytes()).hexdigest()
report['development_pairs'] = len(dev_pairs)
report['model_generation_executed'] = MODEL_FRACTION > 0
atomic_json(OUT / 'task1_validation.json', report)
print('SAVED DATASET VALIDATION:', json.dumps(report, indent=2))
for category in CATEGORIES:
    i = next(i for i,s in enumerate(train_specs) if s['category'] == category)
    print(category, json.dumps(pairs[i]))
RUN_TASKS_2_3 = False  # Keep supplied sections inactive until teammates complete them.


SMOKE: {
  "pair_count": 100,
  "category_counts": {
    "addition": 20,
    "division": 20,
    "algebra": 20,
    "subtraction": 20,
    "multiplication": 20
  },
  "origins": {
    "synthetic": 100
  },
  "negative_types": {
    "wrong_operation": 30,
    "offset_error": 36,
    "sign_error": 24,
    "unhelpful": 10
  },
  "max_length_including_four_newlines": 58,
  "duplicate_questions": 0,
  "duplicate_pairs": 0,
  "unsupported_characters": 0,
  "identical_responses": 0,
  "dev_overlap": 0,
  "model_context_limit": 256,
  "context_validation": "checkpoint header inspected; model forward not executed"
}
Rejected smoke corruption: positive
Rejected smoke corruption: correct_negative
Rejected smoke corruption: question
Rejected smoke corruption: duplicate
Rejected smoke corruption: character
Rejected smoke corruption: length
Representative smoke pairs: [
  {
    "negative": "115+86 =? The answer is 29.",
    "positive": "115+86 =? The answer is 201 because 115+86=201."
  },
  {
    "

## Task 2 and Task 3 handoff

Task 2: complete supplied Steps 5-7. Load `pos_neg_pairs.json` as a list of two-key records. Each string already includes the question; do not prepend it again. Keep four newline terminators and `max_length=64`; retain the no-truncation invariant if changing preprocessing. Use only training pairs; never concatenate `task1_dev_pairs.json`. The supplied zero-token padding shares ID 0 with newline; review masking and preference-loss treatment as part of Task 2. The supplied batch helper drops an incomplete final batch. For 100,000 pairs and batch size 64, 32 records are omitted per epoch; consider including that batch in Task 2. Fix the GPU index assumption in supplied configuration for your hardware. Leave `model.py` and `configurator.py` unchanged.

Task 3: complete Step 8. Use separate unseen equations; compare exact integer answers and inspect explanations. Report performance by category and length/difficulty, show failures and successes, and describe whether development questions informed tuning. Replace the supplied unconditional `.cuda()` with `.to(device)` for CPU/MPS. Do not claim synthetic training negatives were generated by the pretrained model.

Run the supplied sections from the `dpo/` working directory because their paths are relative to that folder. Enable `RUN_TASKS_2_3=True` only once teammates have completed their sections and checked dependencies/paths. These original code cells are guarded so Task 1 can run from top to bottom without starting training, saving an untrained DPO checkpoint, or trying to load a nonexistent trained model. One `pass` preserves the unfinished testing cell's Python syntax.

Regeneration: open this notebook from the project root or `dpo/`, change configuration, and run the Task 1 cells sequentially. Default generation needs no model forward pass. Outputs: training JSON, separate provenance, separate development JSON and validation report. The printed report and saved outputs document checks actually run. Optional NanoGPT generation was not run for the default synthetic dataset.

Checklist: exact schema; 100,000 default pairs; five balanced categories; correct worked positives; varied provably wrong/unhelpful negatives; shared questions; no identical responses/duplicate questions/pairs; vocabulary and full preprocessing length checks; actual checkpoint configuration inspection; fixed seed; progress; optional batched/resumable model generation; development separation; small smoke test then full generation then persisted-file validation. Task 2 is complete; Task 3 remains unfinished.

References: supplied SC3000/CZ3005 Assignment 1 PDF (Task 1 and file conventions); supplied project `pos_neg_pairs.json`, `dpo.ipynb`, `meta.pkl` and checkpoint; [NanoGPT-Math](https://github.com/zhouyuan888888/NanoGPT-Math); [NanoGPT](https://github.com/karpathy/nanoGPT); [Direct Preference Optimization paper](https://arxiv.org/abs/2305.18290) (the Task 2 objective implemented in Step 7). Exact integer computation and Fraction verification are our generation approach; the architecture and supplied preprocessing originate from the base project.

## Supplied notebook sections (Tasks 2 and 3 pending)

### Step 1: Install necesscary packages

In [6]:
if RUN_TASKS_2_3:
    pass  # Supplied placeholder cells may contain comments only.
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib", "torch", "numpy", "transformers", "datasets", "tiktoken", "wandb", "tqdm"])


### Step 2: Package imports and configuration

In [7]:
if RUN_TASKS_2_3:
    pass  # Supplied placeholder cells may contain comments only.
    import sys
    import os
    sys.path.append(os.path.abspath("..")) 
    # Task 2 fix: the supplied notebook pinned GPU index 1. Honour an index the
    # cluster already set, otherwise use the first visible device.
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", os.environ.get("GPU_INDEX", "0"))
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import random
    import pickle
    from model import GPT, GPTConfig
    import random
    from tqdm import tqdm
    import time
    import json
    import matplotlib.pyplot as plt
    # Configuration
    beta = 2.0  # Task 2: multiplies the reward difference, so this equals the
                # supplied hint's division by beta = 0.5. See the Task 2 notes.
    device = ('cuda' if torch.cuda.is_available() else
              'mps' if torch.backends.mps.is_available() else 'cpu')  # Task 2: added MPS
    base_lr = 1e-4
    epochs = 5
    batch_size = 64
    max_length =64
    num_samples = 1
    max_new_tokens = 200
    temperature = 0.8
    top_k = 200
    # tokenizer
    with open("../sft/meta.pkl", "rb") as f:
        meta = pickle.load(f)
    stoi, itos = meta["stoi"], meta["itos"]
    def encode(s): return [stoi[c] for c in s]
    def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [8]:
if RUN_TASKS_2_3:
    pass  # Supplied placeholder cells may contain comments only.
    def compute_logprob(input_ids):
        inputs = input_ids[:, :-1]
        targets = input_ids[:, 1:]
        logits, _ = gpt(inputs, full_seq=True)
        B, T, V = logits.size()
        logits_flat = logits.reshape(-1, V)
        targets_flat = targets.reshape(-1)
        loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
        loss = loss.reshape(B, T)
        attention_mask = (targets != 0).float()
        loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
        return -loss 
    
    def pad_or_truncate(seq, max_length):
        return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))
    
    def get_batches(lines, batch_size):
        random.shuffle(lines)
        #for l in lines:
        #    print(l[1])
        for i in range(0, len(lines), batch_size):
            batch = lines[i:i+batch_size]
            if len(batch) < batch_size:
                continue
            neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
            pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
            neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
            pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
            yield neg_tensor, pos_tensor


### Step 4: Load the pretrained NanoGPT model

In [9]:
if RUN_TASKS_2_3:
    pass  # Supplied placeholder cells may contain comments only.
    ckpt = torch.load("../sft/gpt.pt", map_location=device)
    gptconf = GPTConfig(**ckpt['model_args'])
    gpt = GPT(gptconf)
    state_dict = ckpt['model']
    unwanted_prefix = '_orig_mod.'
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    gpt.load_state_dict(state_dict)
    gpt.to(device).train()


## Task 2 notes: DPO training (Steps 5-7 complete)

Steps 5-7 below are finished. Step 5 loads the Task 1 pairs and builds batches, Step 6 builds the frozen reference policy, the optimizer and the schedule, and Step 7 runs Direct Preference Optimization. `dpo/train_dpo.py` runs the same objective from the command line for cluster jobs (`python dpo/train_dpo.py --help`); Steps 5 and 7 assert that the notebook and the script agree on preprocessing and on the loss, so the two cannot drift apart. Step 8 (Task 3) is still open.

### Objective

Implicit rewards are policy-minus-reference log-probabilities, and the loss is the sigmoid DPO loss of their difference plus the positive-NLL term from the supplied hint:

`loss = -log sigmoid(beta * ((log p(pos) - log p_ref(pos)) - (log p(neg) - log p_ref(neg)))) + lambda_sft * (-log p(pos))`

`beta = 2.0` multiplies here, which is exactly the supplied hint's division by `beta = 0.5`; `lambda_sft = 0.1` keeps the hint's coefficient. The hint compares raw policy log-probabilities with no reference, so nothing anchors the policy to the model it started from and both log-probabilities can drift together. The frozen SFT copy makes the rewards log-ratios as in the DPO paper, at the cost of a second forward pass per batch and a second copy of 8.84M parameters. `train_dpo.py --reference none` reproduces the reference-free variant if we want the ablation in the report.

### Decisions the handoff asked us to make

- **Padding and newline share id 0.** The supplied `compute_logprob` masks with `targets != 0`, which drops every newline, including the four terminators the model must learn to emit, while still scoring the question. Step 5 builds masks from each sequence's true length instead, so the terminators are scored and only padding is skipped. The printed comparison shows the difference on a real record.
- **Only the answer is scored.** Task 1 guarantees the question plus one space is a prefix of both responses, verified here on all 100,000 records, so the question carries no preference signal. Because log-probabilities are length-normalized, the shared prefix would not cancel between the two branches if it were included. `loss_scope = 'full'` restores the supplied behaviour.
- **Length normalization.** Log-probabilities are averaged over scored tokens, as in the supplied helper, so a longer correct answer is not penalized for its length. `length_norm = 'sum'` switches to total log-probability, which needs a much smaller beta.
- **The final batch is kept.** 100,000 pairs at batch size 64 leaves 32 records that the supplied helper discards; `get_batches_task2` yields the short batch, so every record is seen each epoch.
- **GPU index.** The supplied Step 2 pinned `CUDA_VISIBLE_DEVICES = "1"` and chose between CUDA and CPU only. It now honours a `GPU_INDEX` environment variable, defaults to device 0, and falls back to Apple MPS, so the notebook runs unchanged on a laptop and on the cluster.

### Optimizer, schedule and monitoring

AdamW at the supplied `base_lr = 1e-4` with betas (0.9, 0.95), weight decay 0.01 on matrices and none on biases and LayerNorm gains, which is the NanoGPT split. The learning rate warms up linearly over the first 3% of steps and then follows a cosine to 10% of the base rate; gradients are clipped at norm 1.0. Each epoch logs loss, preference accuracy, the implicit reward margin and the gradient norm, then scores the 500 held-out Task 1 development pairs. Those pairs are never trained on and are used only to monitor progress and to keep the best checkpoint, `dpo_best.pt`; Step 8 is where generated answers get graded. Note that Step 4 leaves the policy in `train()` mode with dropout 0.2 while the reference runs in `eval()`, so single-batch ratios are noisy even though the averages are not.

Outputs: `dpo.pt` (last epoch, with `model_state_dict` and `model_args` as Step 8 expects), `dpo_best.pt`, `task2_history.json` and `task2_training_curves.png`.

### Cost and status

At this model size a full epoch over 100,000 pairs takes roughly eight minutes on an Apple M-series GPU and less on a CUDA card, so the default five epochs is well under an hour and does not need the cluster. `TASK2_SMOKE = True` trains on 512 pairs for one epoch into `smoke_dpo.pt` to check the pipeline in seconds without touching the real checkpoint.

The preprocessing, masking, batching, objective, optimizer and schedule were each tested against the real data, and the loop was run end to end on a stand-in model. The run against the real SFT weights is still outstanding because `model.py` and `configurator.py` were never committed to this repository and `sft/gpt.pt` is a Git LFS pointer that needs `git lfs pull`. No numbers from a real DPO run are reported here yet.

### Step 5: Load Data (**students are required to complete this part!**)

In [ ]:
if RUN_TASKS_2_3:
    # Step 5: load the Task 1 pairs and turn them into padded batches.
    # ---- Task 2 configuration (additions to the supplied Step 2 block) -------
    TASK2_SMOKE = False      # True: 512 pairs, 1 epoch, separate checkpoint
    lambda_sft = 0.1         # weight on the positive-NLL term in the supplied hint
    loss_scope = 'response'  # score the answer only; the question is shared
    length_norm = 'mean'     # per-token average log-probability
    weight_decay = 0.01
    warmup_frac = 0.03
    min_lr_ratio = 0.1
    max_grad_norm = 1.0
    grad_accum = 1
    seed = 3000
    TERMINATOR = '\n\n\n\n'  # supplied preprocessing appends four newlines
    PAD_ID = 0               # supplied padding id; also the id of '\n'
    ckpt_out = './smoke_dpo.pt' if TASK2_SMOKE else './dpo.pt'

    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Load data from ./pos_neg_pairs.json. Each string already contains the
    # question, so nothing is prepended.
    with open('./pos_neg_pairs.json') as f:
        lines = json.load(f)
    with open('./task1_dev_pairs.json') as f:
        dev_lines = json.load(f)
    for rec in lines + dev_lines:
        assert set(rec) == {'positive', 'negative'}, f'unexpected schema: {sorted(rec)}'
    # Development pairs are for monitoring only and are never trained on.
    canonical = lambda rec: ''.join(rec['positive'].split('?', 1)[0].split())
    train_questions = {canonical(r) for r in lines}
    leaked = sum(canonical(r) in train_questions for r in dev_lines)
    assert leaked == 0, f'{leaked} development questions appear in training'
    if TASK2_SMOKE:
        lines = lines[:512]
    print(f'{len(lines)} training pairs, {len(dev_lines)} development pairs, '
          f'{len(train_questions)} distinct training questions')

    def prompt_length(pair):
        # Characters shared by both responses: the question plus one space. The
        # tokeniser is char-level, so this count is also a token count.
        question = pair['positive'].split('?', 1)[0] + '?'
        for key in ('positive', 'negative'):
            assert pair[key].startswith(question + ' '), f'question is not a shared prefix: {pair}'
        return len(question) + 1

    def encode_pair(pair, max_length, scope):
        # Masks come from the true length, not from `!= PAD_ID`: padding shares
        # id 0 with '\n', so a value-based mask would also drop the four
        # terminator newlines the model has to learn to emit.
        start = prompt_length(pair) if scope == 'response' else 0
        encoded = []
        for key in ('negative', 'positive'):
            ids = encode(pair[key] + TERMINATOR)
            assert len(ids) <= max_length, f'{len(ids)} > max_length={max_length}: {pair[key]!r}'
            mask = [start <= i < len(ids) for i in range(max_length)]
            encoded.append((ids + [PAD_ID] * (max_length - len(ids)), mask))
        return encoded

    def get_batches_task2(records, batch_size, rng=None):
        # Unlike the supplied helper this keeps the final short batch, so all
        # 100,000 records are seen each epoch instead of dropping 32 of them.
        order = list(range(len(records)))
        if rng is not None:
            rng.shuffle(order)
        for i in range(0, len(order), batch_size):
            chunk = [records[j] for j in order[i:i + batch_size]]
            encoded = [encode_pair(p, max_length, loss_scope) for p in chunk]
            out = []
            for slot in (0, 1):  # 0 = negative, 1 = positive
                out.append(torch.tensor([e[slot][0] for e in encoded], dtype=torch.long, device=device))
                out.append(torch.tensor([e[slot][1] for e in encoded], dtype=torch.bool, device=device))
            yield tuple(out)

    # Preprocessing checks on the real data: the no-truncation invariant, and
    # what the supplied value-based mask would have scored.
    longest = max(len(encode(r[k] + TERMINATOR)) for r in lines + dev_lines
                  for k in ('positive', 'negative'))
    assert longest <= max_length
    print(f'longest preprocessed sequence: {longest} <= max_length={max_length}')
    sample = lines[0]
    (_, _), (pos_ids, pos_mask) = encode_pair(sample, max_length, loss_scope)
    scored = ''.join(itos[i] for i, m in zip(pos_ids, pos_mask) if m)
    assert scored == sample['positive'][prompt_length(sample):] + TERMINATOR
    print(f'scored region of {sample["positive"]!r}: {scored!r}')
    print(f'explicit mask scores {sum(pos_mask)} tokens; the supplied `targets != 0` mask '
          f'would score {sum(i != PAD_ID for i in pos_ids)} and drop every newline')
    batched = sum(b[0].size(0) for b in get_batches_task2(lines, batch_size))
    assert batched == len(lines), (batched, len(lines))
    print(f'batching covers {batched}/{len(lines)} records per epoch')

    # train_dpo.py runs this same objective from the command line; check the
    # two implementations agree so a cluster run matches the notebook.
    import train_dpo
    assert encode_pair(sample, max_length, loss_scope) == \
        train_dpo.encode_pair(sample, stoi, max_length, loss_scope)
    print('notebook preprocessing matches train_dpo.py')

### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [ ]:
if RUN_TASKS_2_3:
    # Step 6: frozen reference policy, AdamW, and a warmup + cosine schedule.
    import math

    # DPO scores the policy against the SFT model it started from, so keep a
    # frozen copy of the weights Step 4 just loaded.
    reference = GPT(GPTConfig(**ckpt['model_args']))
    reference.load_state_dict(gpt.state_dict())
    reference.to(device).eval()
    reference.requires_grad_(False)

    # AdamW with the NanoGPT decay split: matrices decay, biases/norms do not.
    decay = [p for p in gpt.parameters() if p.requires_grad and p.dim() >= 2]
    no_decay = [p for p in gpt.parameters() if p.requires_grad and p.dim() < 2]
    optimizer = torch.optim.AdamW(
        [dict(params=decay, weight_decay=weight_decay),
         dict(params=no_decay, weight_decay=0.0)],
        lr=base_lr, betas=(0.9, 0.95), eps=1e-8)

    batches_per_epoch = math.ceil(len(lines) / batch_size)
    steps_per_epoch = math.ceil(batches_per_epoch / grad_accum)
    total_steps = steps_per_epoch * (1 if TASK2_SMOKE else epochs)
    warmup_steps = max(1, int(total_steps * warmup_frac))

    def lr_factor(step):
        # Linear warmup, then cosine decay to min_lr_ratio of the base lr.
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    trainable = sum(p.numel() for p in gpt.parameters() if p.requires_grad)
    print(f'{trainable/1e6:.2f}M trainable parameters; {len(decay)} decayed tensors, '
          f'{len(no_decay)} undecayed')
    print(f'{batches_per_epoch} batches/epoch, {steps_per_epoch} optimizer steps/epoch, '
          f'{total_steps} total, {warmup_steps} warmup')

### Step 7: Begin training (**students are required to complete this part!**)

In [ ]:
if RUN_TASKS_2_3:
    # Step 7: DPO training against the frozen SFT reference.
    import inspect

    def forward_logits(model, inputs):
        # The supplied helper calls gpt(inputs, full_seq=True); vanilla NanoGPT
        # has no such argument and returns only the last position unless
        # `targets` is given, so fall back to that and discard its loss.
        if 'full_seq' in inspect.signature(type(model).forward).parameters:
            logits, _ = model(inputs, full_seq=True)
        else:
            logits, _ = model(inputs, targets=inputs)
        return logits

    def sequence_logprob(model, ids, mask, norm=None):
        # Log-probability of the scored tokens. This replaces the supplied
        # compute_logprob: it takes the model explicitly (the reference needs
        # it too) and masks by true length rather than by token value.
        norm = norm or length_norm
        inputs, targets = ids[:, :-1], ids[:, 1:]
        logits = forward_logits(model, inputs)
        B, T, V = logits.shape
        token_logprob = -F.cross_entropy(
            logits.reshape(-1, V).float(), targets.reshape(-1), reduction='none').view(B, T)
        scored = mask[:, 1:].to(token_logprob.dtype)
        total = (token_logprob * scored).sum(dim=1)
        if norm == 'sum':
            return total
        return total / scored.sum(dim=1).clamp(min=1.0)

    def dpo_loss(pol_pos, pol_neg, ref_pos, ref_neg):
        # Sigmoid DPO (Rafailov et al. 2023) plus the positive-NLL term from the
        # supplied hint. beta multiplies here, so beta=2.0 is identical to the
        # hint's division by beta=0.5.
        pos_ratio = pol_pos - ref_pos
        neg_ratio = pol_neg - ref_neg
        logits = beta * (pos_ratio - neg_ratio)
        preference = -F.logsigmoid(logits).mean()
        sft = -pol_pos.mean()
        stats = dict(dpo_loss=preference.item(), sft_loss=sft.item(),
                     accuracy=(logits > 0).float().mean().item(),
                     margin=(pos_ratio - neg_ratio).mean().item(),
                     pos_reward=pos_ratio.mean().item(), neg_reward=neg_ratio.mean().item())
        return preference + lambda_sft * sft, stats

    def logprob_pairs(ids_masks):
        neg_ids, neg_mask, pos_ids, pos_mask = ids_masks
        pol_pos = sequence_logprob(gpt, pos_ids, pos_mask)
        pol_neg = sequence_logprob(gpt, neg_ids, neg_mask)
        with torch.no_grad():
            ref_pos = sequence_logprob(reference, pos_ids, pos_mask)
            ref_neg = sequence_logprob(reference, neg_ids, neg_mask)
        return pol_pos, pol_neg, ref_pos, ref_neg

    @torch.no_grad()
    def evaluate_dev(records):
        # Preference accuracy and reward margin on the held-out Task 1 pairs.
        # Monitoring only; Task 3 scores generated answers separately.
        gpt.eval()
        totals, seen = {}, 0
        for batch in get_batches_task2(records, batch_size):
            _, stats = dpo_loss(*logprob_pairs(batch))
            for k, v in stats.items():
                totals[k] = totals.get(k, 0.0) + v
            seen += 1
        gpt.train()
        return {k: v / max(1, seen) for k, v in totals.items()}

    # The two implementations must agree on the objective as well.
    _a, _b, _z = torch.full((4,), -1.0), torch.full((4,), -3.0), torch.zeros(4)
    assert torch.allclose(dpo_loss(_a, _b, _z, _z)[0],
                          train_dpo.dpo_loss(_a, _b, _z, _z, beta, lambda_sft)[0])
    print(f'objective matches train_dpo.py; at initialisation the loss is '
          f'-log sigmoid(0) = {math.log(2):.4f} plus {lambda_sft} x positive NLL')

    use_bf16 = device == 'cuda' and torch.cuda.is_bf16_supported()
    history = {'train': [], 'dev': [], 'log_every': 5 if TASK2_SMOKE else 50}
    best_dev = (-float('inf'), -float('inf'))
    gpt.train()
    for epoch in range(1 if TASK2_SMOKE else epochs):
        rng = random.Random(seed + epoch)
        pbar = tqdm(get_batches_task2(lines, batch_size, rng), total=batches_per_epoch,
                    desc=f'epoch {epoch+1}')
        # Two accumulators: one for the epoch average, one for the logged
        # window. Counts are per key because grad_norm only exists on step
        # boundaries.
        running, running_n, window, window_n = {}, {}, {}, {}
        seen, step, started = 0, 0, time.time()
        optimizer.zero_grad(set_to_none=True)
        for micro, batch in enumerate(pbar):
            ###########################################################
            # DPO step: implicit rewards are policy-minus-reference
            # log-probabilities, and the loss is the log-sigmoid of their
            # difference, scaled by beta.
            ###########################################################
            if use_bf16:
                with torch.autocast('cuda', dtype=torch.bfloat16):
                    pol_pos, pol_neg, ref_pos, ref_neg = logprob_pairs(batch)
            else:
                pol_pos, pol_neg, ref_pos, ref_neg = logprob_pairs(batch)
            loss, stats = dpo_loss(pol_pos, pol_neg, ref_pos, ref_neg)
            (loss / grad_accum).backward()
            if (micro + 1) % grad_accum == 0 or micro + 1 == batches_per_epoch:
                stats['grad_norm'] = float(
                    torch.nn.utils.clip_grad_norm_(gpt.parameters(), max_grad_norm))
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                step += 1
            stats['loss'] = loss.item()
            seen += 1
            for k, v in stats.items():
                running[k] = running.get(k, 0.0) + v
                running_n[k] = running_n.get(k, 0) + 1
                window[k] = window.get(k, 0.0) + v
                window_n[k] = window_n.get(k, 0) + 1
            if seen % history['log_every'] == 0:
                avg = {k: window[k] / window_n[k] for k in window}
                window, window_n = {}, {}
                history['train'].append(dict(epoch=epoch + 1, step=step,
                                             lr=scheduler.get_last_lr()[0], **avg))
                pbar.set_postfix(loss=f'{avg["loss"]:.4f}', acc=f'{avg["accuracy"]:.3f}',
                                 margin=f'{avg["margin"]:.4f}')
        epoch_avg = {k: running[k] / running_n[k] for k in running}
        print(f'epoch {epoch+1}: loss={epoch_avg["loss"]:.4f} dpo={epoch_avg["dpo_loss"]:.4f} '
              f'sft={epoch_avg["sft_loss"]:.4f} acc={epoch_avg["accuracy"]:.3f} '
              f'margin={epoch_avg["margin"]:.4f} ({time.time()-started:.0f}s)')
        dev_stats = evaluate_dev(dev_lines)
        history['dev'].append(dict(epoch=epoch + 1, **dev_stats))
        print(f'  dev: acc={dev_stats["accuracy"]:.3f} margin={dev_stats["margin"]:.4f} '
              f'dpo={dev_stats["dpo_loss"]:.4f}')

        ckpt_path = ckpt_out
        torch.save({
            "model_state_dict": gpt.state_dict(),
            "model_args": ckpt['model_args'],
            "epoch": epoch + 1,
            "train_metrics": epoch_avg,
            "dev_metrics": dev_stats,
        }, ckpt_path)
        print(f"Saved checkpoint to {ckpt_path}")
        # Preference accuracy saturates at 1.000 within an epoch, so break ties
        # on the reward margin; otherwise 'best' just means 'first'.
        score = (dev_stats['accuracy'], dev_stats['margin'])
        if score > best_dev:
            best_dev = score
            best_path = ckpt_path.replace('.pt', '_best.pt')
            torch.save(torch.load(ckpt_path, map_location='cpu', weights_only=False), best_path)
            print(f'  new best development accuracy ({best_dev:.3f}) -> {best_path}')

    with open('./task2_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    if history['train']:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        xs = list(range(len(history['train'])))
        axes[0].plot(xs, [r['loss'] for r in history['train']], label='total')
        axes[0].plot(xs, [r['dpo_loss'] for r in history['train']], label='dpo')
        axes[0].set_title('loss'); axes[0].legend()
        axes[1].plot(xs, [r['accuracy'] for r in history['train']])
        axes[1].set_title('training preference accuracy')
        axes[2].plot(xs, [r['margin'] for r in history['train']])
        axes[2].set_title('implicit reward margin')
        for ax in axes:
            ax.set_xlabel(f'logged step (every {history["log_every"]} batches)')
        fig.tight_layout()
        fig.savefig('./task2_training_curves.png', dpi=120)
        plt.show()
    print('Task 2 complete. Step 8 (Task 3) loads ./dpo.pt or ./dpo_best.pt.')

### Step 8: Begin testing (**students are required to complete this part!**)

In [13]:
if RUN_TASKS_2_3:
    pass  # Supplied placeholder cells may contain comments only.
    # Load the fine-tuned model
    ckpt_path = "../dpo/dpo.pt"
    checkpoint = torch.load(ckpt_path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    gpt = GPT(gptconf).cuda()
    try:
        state_dict = checkpoint['model']
    except:
        state_dict = checkpoint['model_state_dict']
    unwanted_prefix = '_orig_mod.'
    for k,v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    gpt.load_state_dict(state_dict)
    # Test
    gpt.eval()
    test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
    with torch.no_grad():
        for prompt in test_set: 
            prompt_ids = encode(prompt)
            ###########################################################
            # Please complete the test code here!
            # ...
            # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
            # ...
            ###########################################################
